# NB_Silver_Orders — Bronze to Silver Transformation (Collaborative Deep-Dive & Upgrade)

## Section 1 — Purpose, Business Context & Collaborative Review Legend

- **Source Table:** `Cloud_Retail_Data_Lakehouse_v2.dbo.orders` → ADLS Gen2 / OneLake Shortcut (`Files/bronze/sales/orders/`).
- **Silver Target:** `silver.orders` (Managed Delta Table) + `silver._quarantine_orders` (Conditional Quarantine).
- **Grain:** One row per customer order — primary key `order_id` (`99,441` rows). Each `order_id` maps `1:1` to a distinct `customer_id` (`99,441` unique `customer_id`s in `silver.customers`).
- **Verified Bronze Profile (`99,441` rows × `8` columns):**
  - `0` duplicate `order_id`s, `0` duplicate `customer_id`s, `0` invalid 32-char hex IDs, `0` unparseable timestamps.
  - **Status Distribution (`8` official values):** `delivered` (`96,478`), `shipped` (`1,107`), `canceled` (`625`), `unavailable` (`609`), `invoiced` (`314`), `processing` (`301`), `created` (`5`), `approved` (`2`).
  - **Legitimate Lifecycle NULLs:** `order_approved_at` (`160` NULLs), `order_delivered_carrier_date` (`1,783` NULLs), `order_delivered_customer_date` (`2,965` NULLs). Never coalesce or fabricate these dates.
  - **Temporal & Completeness Anomalies (Flagged, NOT Dropped):**
    - `23` orders have `order_delivered_customer_date < order_delivered_carrier_date` (`delivered` before carrier handoff).
    - `166` orders have `order_delivered_carrier_date < order_purchase_timestamp`; `1,359` have `carrier < approved`; `61` have `delivered_customer < approved`.
    - `14` `delivered` orders lack `order_approved_at`, `2` lack `carrier_date`, `8` lack `customer_date`. Conversely, `6` `canceled` orders still have a populated `order_delivered_customer_date`.
    - `7,827` orders were delivered after `order_estimated_delivery_date` (`is_late_delivery = True`) vs `88,649` on time.
  - **Cross-Table Integrity:** `0` orphan `customer_id`s vs `silver.customers`; `775` orders (`603 unavailable`, `164 canceled`, `5 created`, `2 invoiced`, `1 shipped`) have zero rows in `order_items` (`LEFT JOIN` required in Gold).

### Collaborative Code-Review Legend
This notebook preserves the original teammate draft (`Orders.ipynb`) alongside detailed explanations and production upgrades:
- **`[KEPT / BUILT UPON]`**: Sound engineering patterns from the original notebook (direct Spark Parquet read, required-column contract, PK uniqueness check, `silver_processed_at` lineage, and post-write read-back check).
- **`[ORIGINAL — COMMENTED FOR REVIEW]`**: Original teammate code preserved verbatim inside comments (inactive so `Run All` never fails) with a detailed explanation of why specific copy-pasted transforms (`lpad(..., 5, "0")` on `order_status` and `trim`/`upper` on `timestamp` columns) silently corrupted data.
- **`[IMPROVED REPLACEMENT]`**: Clean, type-safe PySpark replacement directly below the commented cell.
- **`[NEW ADDITION — PRODUCTION STANDARD]`**: Added enterprise controls (`WRITE_ENABLED = False`, `silver.customers` FK check, Quarantine routing, chronology & SLA flags, full lineage, and reconciliation).

> **WARNING — SAFETY LOCK ENABLED (`WRITE_ENABLED = False`):** Do not enable writes until a Dry Run completes cleanly and all reconciliations pass.

## Section 2 — Configuration & Safety Controls `[NEW ADDITION — PRODUCTION STANDARD]`

Centralizes all paths, valid domain values, table names, lineage metadata, and the `WRITE_ENABLED = False` safety switch. Notice we target **`silver.orders`** (instead of `silver.orders_poc_delta`) to match `silver.customers`, `silver.sellers`, `silver.geolocation_zip_reference`, and `silver.order_items`.

In [ ]:
# [NEW ADDITION — PRODUCTION STANDARD] Section 2: Configuration & Safety Controls
from pyspark.sql import functions as F

WRITE_ENABLED = False          # SAFETY LOCK: flip to True only after Dry Run review
RUN_CUSTOMER_FK_CHECK = True   # Cross-check customer_id against silver.customers when present

BRONZE_PATH = "Files/bronze/sales/orders/orders_20261121_013142_b0e93f82-84a0-4668-9a8c-9ae1da7dd183.parquet"

SILVER_SCHEMA = "silver"
SILVER_TARGET_TABLE = f"{SILVER_SCHEMA}.orders"
QUARANTINE_TARGET_TABLE = f"{SILVER_SCHEMA}._quarantine_orders"
SILVER_CUSTOMERS_TABLE = f"{SILVER_SCHEMA}.customers"

INGESTION_RUN_ID = "b0e93f82-84a0-4668-9a8c-9ae1da7dd183"
SOURCE_SYSTEM = "sqlserver_olist"
SOURCE_TABLE = "dbo.orders"

VALID_STATUSES = [
    "delivered", "shipped", "canceled", "unavailable",
    "invoiced", "processing", "created", "approved"
]

EXPECTED_BRONZE_COLS = [
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

print(f"Configured BRONZE_PATH        : {BRONZE_PATH}")
print(f"Configured SILVER_TARGET      : {SILVER_TARGET_TABLE}")
print(f"Configured QUARANTINE_TARGET  : {QUARANTINE_TARGET_TABLE}")
print(f"RUN_CUSTOMER_FK_CHECK         : {RUN_CUSTOMER_FK_CHECK}")
print(f"WRITE_ENABLED                 : {WRITE_ENABLED}")

## Section 3 — Distributed Bronze Parquet Ingestion `[KEPT / BUILT UPON]`

**Why we kept the teammate's ingestion pattern:** Unlike earlier Pandas drafts, `Orders.ipynb` correctly read Bronze Parquet directly via Spark (`spark.read.format("parquet").load(BRONZE_PATH)`) using the relative `Files/bronze/...` shortcut path. This preserves native `timestamp` schemas for all 5 date columns right out of Parquet! We simply add `_source_file` via `F.input_file_name()` for file-level lineage.

In [ ]:
# [KEPT / BUILT UPON] Section 3: Native Spark Parquet Ingestion + _source_file Lineage
df_bronze_raw = (
    spark.read
    .format("parquet")
    .load(BRONZE_PATH)
    .withColumn("_source_file", F.input_file_name())
)

bronze_count = df_bronze_raw.count()
print(f"Bronze Orders Row Count: {bronze_count:,}")
df_bronze_raw.printSchema()
display(df_bronze_raw.limit(5))

## Stage 1 — Schema Contract & Raw Null Baseline `[KEPT / BUILT UPON]`

Keeps the teammate's `missing_columns` contract check, adds `extra_cols` schema-drift detection, and profiles raw `NULL` counts across all 8 columns. Remember: `NULL`s in `order_approved_at` (`160`), `order_delivered_carrier_date` (`1,783`), and `order_delivered_customer_date` (`2,965`) are expected for non-delivered/canceled orders.

In [ ]:
# [KEPT / BUILT UPON] Stage 1: Schema Contract & Raw Null Summary
actual_data_cols = [c for c in df_bronze_raw.columns if c != "_source_file"]
missing_cols = [c for c in EXPECTED_BRONZE_COLS if c not in actual_data_cols]
extra_cols = [c for c in actual_data_cols if c not in EXPECTED_BRONZE_COLS]

if missing_cols:
    raise ValueError(f"Stage 1 FAILED — Missing required Bronze columns: {missing_cols}")

if extra_cols:
    print(f"WARNING — Extra columns observed in Bronze (excluded from Silver contract): {extra_cols}")

df_bronze = df_bronze_raw.select(*EXPECTED_BRONZE_COLS, "_source_file")

raw_nulls = df_bronze.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"{c}_nulls")
    for c in EXPECTED_BRONZE_COLS
])
print("Stage 1 Schema Validation: PASSED")
display(raw_nulls)

## Stage 2A — Teammate Original Cleaning & Standardization `[ORIGINAL — COMMENTED FOR REVIEW]`

**What was RIGHT (`[KEPT / BUILT UPON]`):**
- Checked `required_columns`, filtered empty/null `order_id`, checked `order_id` primary-key uniqueness, and stamped `silver_processed_at`.

**Three subtle copy-paste bugs from a `customers`/`sellers` template that silently corrupted `silver.orders_poc_delta`:**
1. **`F.lpad(col("order_status"), 5, "0")` (Copied from `zip_code_prefix`!):** `lpad(..., 5, "0")` forces every string to **exact length 5** — so any status longer than 5 characters is **truncated**: `"delivered"` becomes `"deliv"`, `"shipped"` becomes `"shipp"`, `"canceled"` becomes `"cance"`, `"unavailable"` becomes `"unava"`!
2. **`F.trim(order_purchase_timestamp)` & `F.upper(F.trim(order_approved_at))` (Copied from `state`!):** Even though Bronze Parquet already gave us native `timestamp` types, calling string functions `trim()` and `upper()` on a `timestamp` column **implicitly converts it back to `string`**! (Visible in the original Cell [5] output: `order_purchase_timestamp: string`, `order_approved_at: string`.)
3. **Silent `dropDuplicates()` & typo in error message:** Unaudited `dropDuplicates()` before measuring duplicates, and the `order_id` error message read `"customer_id is not unique."`.

In [ ]:
# ==============================================================================
# [ORIGINAL TEAMMATE CELL 2 — COMMENTED FOR REVIEW]
# Good: Schema check, PK uniqueness check, silver_processed_at lineage.
# Replaced: Fixed lpad(5) truncation on order_status & string downgrade on timestamps.
# ==============================================================================
# invalid_orders_count = df_bronze.filter(
#     F.col("order_id").isNull() | (F.length(F.trim(F.col("order_id").cast("string"))) == 0)
# ).count()
#
# df_clean = (
#     df_bronze
#     .withColumn("order_id", F.trim(F.col("order_id").cast("string")))
#     .withColumn("customer_id", F.trim(F.col("customer_id").cast("string")))
#     .withColumn("order_status", F.lpad(F.col("order_status").cast("string"), 5, "0"))  # BUG: truncates "delivered" -> "deliv"!
#     .withColumn("order_purchase_timestamp", F.trim(F.col("order_purchase_timestamp")))    # BUG: casts timestamp -> string!
#     .withColumn("order_delivered_carrier_date", F.to_timestamp(F.trim(F.col("order_delivered_carrier_date")), "yyyy-MM-dd HH:mm:ss"))
#     .withColumn("order_delivered_customer_date", F.to_timestamp(F.trim(F.col("order_delivered_customer_date")), "yyyy-MM-dd HH:mm:ss"))
#     .withColumn("order_estimated_delivery_date", F.to_timestamp(F.trim(F.col("order_estimated_delivery_date")), "yyyy-MM-dd HH:mm:ss"))
#     .withColumn("order_approved_at", F.upper(F.trim(F.col("order_approved_at"))))        # BUG: casts timestamp -> string!
# )
#
# df_clean = df_clean.filter(F.col("order_id").isNotNull() & (F.length(F.col("order_id")) > 0))
# df_clean = df_clean.dropDuplicates()
# duplicate_keys = df_clean.groupBy("order_id").count().filter(F.col("count") > 1)
# if duplicate_keys.count() > 0:
#     raise ValueError("Data quality failed: customer_id is not unique.")
# df_silver = df_clean.withColumn("silver_processed_at", F.current_timestamp())
print("Skipped commented original Cell 2 — executing type-safe Silver standardization next.")

## Stage 2B — Type-Safe Standardization & Status Normalization `[IMPROVED REPLACEMENT]`

1. Normalizes `order_id`, `customer_id`, and `order_status` using `F.lower(F.trim(...))` (never `lpad`).
2. Preserves native `TIMESTAMP` types for **all 5 lifecycle columns** using `try_cast(... AS TIMESTAMP)` and tracks `_cast_failed` if any non-null raw value fails conversion.
3. Measures `exact_dup_count` explicitly without silently dropping rows.

In [ ]:
# [IMPROVED REPLACEMENT] Stage 2B: Type-Safe Standardization & Timestamp Preservation
df_pre = (
    df_bronze
    .withColumn("_raw_purchase_str", F.col("order_purchase_timestamp").cast("string"))
    .withColumn("_raw_approved_str", F.col("order_approved_at").cast("string"))
    .withColumn("_raw_carrier_str", F.col("order_delivered_carrier_date").cast("string"))
    .withColumn("_raw_customer_str", F.col("order_delivered_customer_date").cast("string"))
    .withColumn("_raw_estimated_str", F.col("order_estimated_delivery_date").cast("string"))
)

df_std = (
    df_pre
    .withColumn("order_id", F.lower(F.trim(F.col("order_id").cast("string"))))
    .withColumn("customer_id", F.lower(F.trim(F.col("customer_id").cast("string"))))
    .withColumn("order_status", F.lower(F.trim(F.col("order_status").cast("string"))))
    .withColumn("order_purchase_timestamp", F.expr("try_cast(order_purchase_timestamp AS TIMESTAMP)"))
    .withColumn("order_approved_at", F.expr("try_cast(order_approved_at AS TIMESTAMP)"))
    .withColumn("order_delivered_carrier_date", F.expr("try_cast(order_delivered_carrier_date AS TIMESTAMP)"))
    .withColumn("order_delivered_customer_date", F.expr("try_cast(order_delivered_customer_date AS TIMESTAMP)"))
    .withColumn("order_estimated_delivery_date", F.expr("try_cast(order_estimated_delivery_date AS TIMESTAMP)"))
    .withColumn("_cast_failed", (
        (F.col("_raw_purchase_str").isNotNull() & F.col("order_purchase_timestamp").isNull()) |
        (F.col("_raw_approved_str").isNotNull() & F.col("order_approved_at").isNull()) |
        (F.col("_raw_carrier_str").isNotNull() & F.col("order_delivered_carrier_date").isNull()) |
        (F.col("_raw_customer_str").isNotNull() & F.col("order_delivered_customer_date").isNull()) |
        (F.col("_raw_estimated_str").isNotNull() & F.col("order_estimated_delivery_date").isNull())
    ))
)

exact_dup_count = bronze_count - df_bronze.dropDuplicates(EXPECTED_BRONZE_COLS).count()
cast_failures = df_std.filter(F.col("_cast_failed")).count()
print(f"Stage 2B Standardization complete. Exact duplicates: {exact_dup_count}, Cast failures: {cast_failures}")
df_std.select(*EXPECTED_BRONZE_COLS).printSchema()
display(df_std.groupBy("order_status").count().orderBy(F.desc("count")))

## Stage 3 — Primary Key (`order_id`) & `1:1` `customer_id` Uniqueness Check `[KEPT / BUILT UPON + NEW ADDITION]`

Verifies both `order_id` (Primary Key) and `customer_id` (1:1 per-order session key in Olist) have `0` duplicates before validation.

In [ ]:
# [KEPT / BUILT UPON + NEW ADDITION] Stage 3: PK & 1:1 customer_id Uniqueness
dup_order_df = (
    df_std.filter(F.col("order_id").isNotNull())
    .groupBy("order_id").count().filter(F.col("count") > 1)
    .select("order_id").withColumn("_dup_order_pk", F.lit(True))
)
dup_order_cnt = dup_order_df.count()
dup_cust_cnt = df_std.filter(F.col("customer_id").isNotNull()).groupBy("customer_id").count().filter(F.col("count") > 1).count()

df_std_pk = (
    df_std
    .join(dup_order_df, on="order_id", how="left")
    .withColumn("_dup_order_pk", F.coalesce(F.col("_dup_order_pk"), F.lit(False)))
)

print(f"Duplicate order_id PKs    : {dup_order_cnt} (Expected: 0)")
print(f"Duplicate customer_id FKs : {dup_cust_cnt} (Expected: 0 — 1:1 with order_id)")

## Stage 4 — Domain Validation, `silver.customers` FK Check & Quarantine `[NEW ADDITION — PRODUCTION STANDARD]`

Routes any invalid record to `df_quarantine` with an explicit `dq_rejection_reason` instead of silently dropping rows:
1. Valid 32-char lowercase hex format (`^[0-9a-f]{32}$`) for `order_id` and `customer_id`.
2. `order_status` belongs to the 8 official Olist statuses (`VALID_STATUSES`).
3. Mandatory timestamps (`order_purchase_timestamp`, `order_estimated_delivery_date`) are non-null, with `_cast_failed = False` and `_dup_order_pk = False`.
4. Optional `left_anti` Referential Integrity check against `silver.customers` (`RUN_CUSTOMER_FK_CHECK = True`).

In [ ]:
# [NEW ADDITION — PRODUCTION STANDARD] Stage 4: Validation, Customer FK Check & Quarantine
hex32_regex = r"^[0-9a-f]{32}$"

if RUN_CUSTOMER_FK_CHECK and spark.catalog.tableExists(SILVER_CUSTOMERS_TABLE):
    df_cust_ref = spark.table(SILVER_CUSTOMERS_TABLE).select("customer_id").distinct()
    missing_cust_df = (
        df_std_pk.select("customer_id").distinct()
        .join(df_cust_ref, on="customer_id", how="left_anti")
        .withColumn("_missing_customer_fk", F.lit(True))
    )
    missing_cust_cnt = missing_cust_df.count()
    df_checked = (
        df_std_pk
        .join(missing_cust_df, on="customer_id", how="left")
        .withColumn("_missing_customer_fk", F.coalesce(F.col("_missing_customer_fk"), F.lit(False)))
    )
    print(f"Referential Check vs {SILVER_CUSTOMERS_TABLE}: {missing_cust_cnt} unmatched customer_id(s) (Expected: 0)")
else:
    df_checked = df_std_pk.withColumn("_missing_customer_fk", F.lit(False))
    print(f"Referential Check vs {SILVER_CUSTOMERS_TABLE} skipped (table not found or check disabled).")

df_flagged = df_checked.withColumn(
    "dq_rejection_reason",
    F.concat_ws(
        " | ",
        F.when(~F.coalesce(F.col("order_id"), F.lit("")).rlike(hex32_regex), F.lit("INVALID_ORDER_ID_HEX32")),
        F.when(~F.coalesce(F.col("customer_id"), F.lit("")).rlike(hex32_regex), F.lit("INVALID_CUSTOMER_ID_HEX32")),
        F.when(~F.coalesce(F.col("order_status"), F.lit("")).isin(VALID_STATUSES), F.lit("INVALID_ORDER_STATUS")),
        F.when(F.col("order_purchase_timestamp").isNull(), F.lit("NULL_PURCHASE_TIMESTAMP")),
        F.when(F.col("order_estimated_delivery_date").isNull(), F.lit("NULL_ESTIMATED_DELIVERY_DATE")),
        F.when(F.col("_cast_failed"), F.lit("TIMESTAMP_CAST_FAILED")),
        F.when(F.col("_dup_order_pk"), F.lit("DUPLICATE_ORDER_ID_PK")),
        F.when(F.col("_missing_customer_fk"), F.lit("MISSING_CUSTOMER_FK")),
    )
)

df_valid = df_flagged.filter(F.length(F.col("dq_rejection_reason")) == 0)
df_quarantine = (
    df_flagged.filter(F.length(F.col("dq_rejection_reason")) > 0)
    .select(*EXPECTED_BRONZE_COLS, "dq_rejection_reason", "_source_file")
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("source_table", F.lit(SOURCE_TABLE))
    .withColumn("ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumn("quarantined_at", F.current_timestamp())
)

valid_count = df_valid.count()
quarantine_count = df_quarantine.count()
print(f"Valid Order Rows : {valid_count:,} (Expected: 99,441)")
print(f"Quarantined Rows : {quarantine_count:,} (Expected: 0)")

## Stage 5 — Temporal Chronology & Delivery SLA Diagnostics `[NEW ADDITION — PRODUCTION STANDARD]`

Real-world Olist timestamps contain minor chronological inversions and missing intermediate dates on `23`–`1,359` orders. Instead of dropping valid revenue orders, we expose transparent analytical flags for Gold SLA modeling:
- `has_delivery_chronology_anomaly`: `True` when `delivered_customer < delivered_carrier` (`23`), `delivered_customer < approved_at` (`61`), or `delivered_carrier < purchase_timestamp` (`166`).
- `is_delivered_missing_timestamps`: `True` when `order_status == 'delivered'` yet any of `approved_at` (`14`), `carrier_date` (`2`), or `customer_date` (`8`) is `NULL`.
- `is_late_delivery`: `order_delivered_customer_date > order_estimated_delivery_date` (`7,827` late vs `88,649` on-time; `NULL` when not yet delivered).
- `delivery_lead_time_days`: Actual end-to-end delivery duration in days (`ROUND((unix_timestamp(delivered) - unix_timestamp(purchase)) / 86400.0, 2)`).

In [ ]:
# [NEW ADDITION — PRODUCTION STANDARD] Stage 5: Chronology Flags & Delivery SLA Metrics
df_enriched = (
    df_valid
    .withColumn(
        "has_delivery_chronology_anomaly",
        (
            (F.col("order_delivered_customer_date").isNotNull() & F.col("order_delivered_carrier_date").isNotNull() &
             (F.col("order_delivered_customer_date") < F.col("order_delivered_carrier_date"))) |
            (F.col("order_delivered_customer_date").isNotNull() & F.col("order_approved_at").isNotNull() &
             (F.col("order_delivered_customer_date") < F.col("order_approved_at"))) |
            (F.col("order_delivered_carrier_date").isNotNull() & F.col("order_purchase_timestamp").isNotNull() &
             (F.col("order_delivered_carrier_date") < F.col("order_purchase_timestamp")))
        )
    )
    .withColumn(
        "is_delivered_missing_timestamps",
        (F.col("order_status") == F.lit("delivered")) & (
            F.col("order_approved_at").isNull() |
            F.col("order_delivered_carrier_date").isNull() |
            F.col("order_delivered_customer_date").isNull()
        )
    )
    .withColumn(
        "is_late_delivery",
        F.when(
            F.col("order_delivered_customer_date").isNotNull() & F.col("order_estimated_delivery_date").isNotNull(),
            F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date")
        ).otherwise(F.lit(None).cast("boolean"))
    )
    .withColumn(
        "delivery_lead_time_days",
        F.when(
            F.col("order_delivered_customer_date").isNotNull() & F.col("order_purchase_timestamp").isNotNull(),
            F.round((F.unix_timestamp("order_delivered_customer_date") - F.unix_timestamp("order_purchase_timestamp")) / 86400.0, 2)
        ).otherwise(F.lit(None).cast("double"))
    )
)

diag = df_enriched.agg(
    F.sum(F.when(F.col("order_delivered_customer_date") < F.col("order_delivered_carrier_date"), 1).otherwise(0)).alias("deliv_before_carrier"),
    F.sum(F.when(F.col("order_delivered_carrier_date") < F.col("order_purchase_timestamp"), 1).otherwise(0)).alias("carrier_before_purchase"),
    F.sum(F.when(F.col("has_delivery_chronology_anomaly"), 1).otherwise(0)).alias("any_chronology_anomaly"),
    F.sum(F.when(F.col("is_delivered_missing_timestamps"), 1).otherwise(0)).alias("delivered_missing_ts"),
    F.sum(F.when(F.col("is_late_delivery") == True, 1).otherwise(0)).alias("late_deliveries"),
    F.sum(F.when(F.col("is_late_delivery") == False, 1).otherwise(0)).alias("ontime_deliveries"),
).collect()[0]

print(f"delivered < carrier        : {diag['deliv_before_carrier']} (Expected: 23)")
print(f"carrier < purchase         : {diag['carrier_before_purchase']} (Expected: 166)")
print(f"Any chronology anomaly     : {diag['any_chronology_anomaly']}")
print(f"Delivered missing any TS   : {diag['delivered_missing_ts']}")
print(f"Late vs On-Time deliveries : {diag['late_deliveries']:,} late / {diag['ontime_deliveries']:,} on-time (Expected: 7,827 / 88,649)")

## Stage 6 — Full Lineage & Row Conservation Reconciliation `[NEW ADDITION — PRODUCTION STANDARD]`

Projects the clean `16`-column Silver contract (`8` core columns + `4` SLA/chronology columns + `4` standardized lineage columns) and asserts `bronze_count == silver_count + quarantine_count`.

In [ ]:
# [NEW ADDITION — PRODUCTION STANDARD] Stage 6: Lineage & Reconciliation
df_silver = (
    df_enriched
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("source_table", F.lit(SOURCE_TABLE))
    .withColumn("ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumn("silver_processed_at", F.current_timestamp())
    .select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "has_delivery_chronology_anomaly",
        "is_delivered_missing_timestamps",
        "is_late_delivery",
        "delivery_lead_time_days",
        "source_system",
        "source_table",
        "ingestion_run_id",
        "silver_processed_at",
    )
)

silver_count = df_silver.count()
assert bronze_count == silver_count + quarantine_count, (
    f"Reconciliation mismatch: bronze={bronze_count}, silver={silver_count}, quarantine={quarantine_count}"
)
df_silver.printSchema()
print(f"Stage 6 Reconciliation PASSED: bronze={bronze_count:,}, silver={silver_count:,}, quarantine={quarantine_count:,}")

## Section 5A — Teammate Original Write Cell `[ORIGINAL — COMMENTED FOR REVIEW]`

- **What was RIGHT:** Wrote directly as Delta into the `silver` schema with `overwriteSchema = true`.
- **Why it is replaced below:**
  1. Target name `silver.orders_poc_delta` is a temporary POC table name rather than canonical `silver.orders`.
  2. Missing `WRITE_ENABLED = False` safety lock and `CREATE SCHEMA IF NOT EXISTS silver`.

In [ ]:
# ==============================================================================
# [ORIGINAL TEAMMATE CELL 3 — COMMENTED FOR REVIEW]
# Good: Delta format inside silver schema.
# Replaced: Canonical target silver.orders + WRITE_ENABLED safety gate.
# ==============================================================================
# TARGET_TABLE = "silver.orders_poc_delta"
#
# (
#     df_silver
#     .write
#     .format("delta")
#     .mode("overwrite")
#     .option("overwriteSchema", "true")
#     .saveAsTable(TARGET_TABLE)
# )
#
# print(f"Successfully written to {TARGET_TABLE}")
print("Skipped commented original Cell 3 — executing guarded write to silver.orders next.")

## Section 5B — Guarded Delta Write (`silver.orders`) `[IMPROVED REPLACEMENT]`

Creates `silver` schema if missing and persists `silver.orders` (plus `silver._quarantine_orders` only if `quarantine_count > 0`) strictly when `WRITE_ENABLED = True`.

In [ ]:
# [IMPROVED REPLACEMENT] Section 5B: Guarded Delta Write into silver.orders
if WRITE_ENABLED:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")
    df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER_TARGET_TABLE)
    print(f"Wrote {silver_count:,} rows to {SILVER_TARGET_TABLE}")

    if quarantine_count > 0:
        df_quarantine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(QUARANTINE_TARGET_TABLE)
        print(f"Wrote {quarantine_count:,} rows to {QUARANTINE_TARGET_TABLE}")
    else:
        print("Zero quarantined rows — skipping quarantine table write.")
else:
    print("WRITE_ENABLED is False — dry validation only; no Silver or Quarantine data was written.")

## Section 6 — Post-Write Delta Read-Back Verification `[KEPT / BUILT UPON]`

Expands the teammate's original Cell [5] read-back check (`spark.table(...)`, `printSchema()`, `count()`) to also assert:
1. All `5` date columns remain true `timestamp` types (never `string`).
2. All `8` `order_status` values remain full-length (never truncated by `lpad`).
3. `0` duplicate `order_id`s and exact `99,441` row parity.

In [ ]:
# [KEPT / BUILT UPON] Section 6: Post-Write Validation
if WRITE_ENABLED:
    df_check = spark.table(SILVER_TARGET_TABLE)
    chk_count = df_check.count()
    chk_dup = df_check.groupBy("order_id").count().filter(F.col("count") > 1).count()
    bad_status = df_check.filter(~F.col("order_status").isin(VALID_STATUSES)).count()
    ts_types = {f.name: f.dataType.simpleString() for f in df_check.schema.fields if "timestamp" in f.name or "_at" in f.name or "_date" in f.name}

    assert chk_count == silver_count, f"Row mismatch: {chk_count} vs {silver_count}"
    assert chk_dup == 0, f"Duplicate order_id found: {chk_dup}"
    assert bad_status == 0, f"Invalid/truncated order_status found: {bad_status}"
    assert all(t == "timestamp" for t in ts_types.values()), f"Non-timestamp date column: {ts_types}"

    print(f"Post-write validation PASSED — rows={chk_count:,}, dup_pk={chk_dup}, invalid_status={bad_status}")
    print("Timestamp column types verified:", ts_types)
    display(df_check.groupBy("order_status").count().orderBy(F.desc("count")))
else:
    print("Post-write validation skipped because WRITE_ENABLED is False.")